In [1]:
from qiskit_optimization.algorithms import WarmStartQAOAOptimizer
from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.optimizers import COBYLA
from qiskit_optimization.algorithms import SlsqpOptimizer

from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit.primitives import Sampler

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


def get_warmstart_qaoa_circuit(problem: QuadraticProgram, p=1):
    """
    Generate a warm-started QAOA circuit for a given Qiskit Optimization problem.
    Returns the circuit (not optimized, just warm-started).
    """
    optimizer = COBYLA(maxiter=0)
    sampler = Sampler()
    qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=p)
    ws_qaoa = WarmStartQAOAOptimizer(
        pre_solver=SlsqpOptimizer(),
        relax_for_pre_solver=True,
        qaoa=qaoa,
        epsilon=0.0
    )
    qp2qubo = QuadraticProgramToQubo()
    qubo = qp2qubo.convert(problem)
    result = ws_qaoa.solve(qubo)
    qaoa_circuit = ws_qaoa._qaoa.ansatz
    return qaoa_circuit


In [2]:
from qiskit_optimization import QuadraticProgram
from qiskit.quantum_info import SparsePauliOp

def hamiltonian_to_quadratic_program(hamiltonian: SparsePauliOp):
    n = hamiltonian.num_qubits
    qp = QuadraticProgram()
    for i in range(n):
        qp.binary_var(f'x{i}')
    linear = {f'x{i}': 0.0 for i in range(n)}
    quadratic = {}
    constant = 0.0

    for pauli_str, coeff in zip(hamiltonian.paulis.to_labels(), hamiltonian.coeffs):
        coeff = float(coeff)
        z_indices = [i for i, p in enumerate(pauli_str[::-1]) if p == 'Z']
        if len(z_indices) == 0:
            constant += coeff
        elif len(z_indices) == 1:
            i = z_indices[0]
            # Z_i = 1 - 2x_i
            constant += coeff * 1
            linear[f'x{i}'] += -2 * coeff
        elif len(z_indices) == 2:
            i, j = z_indices
            # Z_i Z_j = 1 - 2x_i - 2x_j + 4x_i x_j
            constant += coeff * 1
            linear[f'x{i}'] += -2 * coeff
            linear[f'x{j}'] += -2 * coeff
            quadratic[(f'x{i}', f'x{j}')] = quadratic.get((f'x{i}', f'x{j}'), 0.0) + 4 * coeff
        else:
            # More than 2 Zs or contains X/Y: skip (not QUBO)
            continue

    qp.minimize(constant=constant, linear=linear, quadratic=quadratic)
    return qp

In [3]:
import json
from qiskit.qasm3 import loads, dumps

circuits = None

with open("quantum_circuits_output_20250911_144036_rl_quantum_4b.json") as f:
    circuits = json.load(f)["results"]

In [4]:
from datasets import load_dataset
DATASET_NAME = "valterUo/graph-data-quantum-optimized"
ds = load_dataset(DATASET_NAME)
rewards = {}

# Take test set and create a new dictionary with signature field as a key
test_dict = list(ds["test"])
for circuit in circuits:
    if not test_dict[circuit["sample_index"]]["signature"] == circuit["signature"]:
        print("Warning: signature mismatch!")

c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
import numpy as np
from reward import KL_divergence_reward, expectation_value_reward
from concurrent.futures import ThreadPoolExecutor

from utils import construct_qiskit_hamiltonian


with open("circuit_rewards.json") as f:
    rewards = json.load(f)

# Calculate and print rewards for each circuit
for j, circuit in enumerate(circuits):
    try:
        cost_hamiltonian_str = circuit['dataset_metrics']['cost_hamiltonian']
        cost_ham = construct_qiskit_hamiltonian(cost_hamiltonian_str)
        qp = hamiltonian_to_quadratic_program(cost_ham)
        warmstart_circuit = get_warmstart_qaoa_circuit(qp, p=1)
        param_map = warmstart_circuit.parameters
        ground_truth_index = circuit["sample_index"]
        test_item = test_dict[ground_truth_index]
        divergences = []

        def compute_divergence():
            param_map_random = {param: np.random.uniform(-np.pi, np.pi) for param in param_map.data}
            new_parametrized_circuit = warmstart_circuit.assign_parameters(param_map_random)
            qasm_str = dumps(new_parametrized_circuit)
            return KL_divergence_reward(qasm_str, test_item)
        
        def compute_expectatation_value():
            param_map_random = {param: np.random.uniform(-np.pi, np.pi) for param in param_map.data}
            new_parametrized_circuit = warmstart_circuit.assign_parameters(param_map_random)
            qasm_str = dumps(new_parametrized_circuit)
            return expectation_value_reward(qasm_str, test_item)

        with ThreadPoolExecutor(max_workers=os.cpu_count()- 1) as executor:
            futures = [executor.submit(compute_divergence) for _ in range(100)]
            divergences = [future.result() for future in futures]

        expectation_values = []

        with ThreadPoolExecutor(max_workers=os.cpu_count()- 1) as executor:
            futures = [executor.submit(compute_expectatation_value) for _ in range(100)]
            expectation_values = [future.result() for future in futures]

        reward = float(np.mean(divergences))
        expec_reward = float(np.mean(expectation_values))
        rewards[str(ground_truth_index)]["warm_start_qaoa_reward"] = reward
        rewards[str(ground_truth_index)]["warm_start_qaoa_expectation_value"] = expec_reward
        print(f"Circuit {j} ws reward: {reward}", rewards[str(ground_truth_index)], f"expectation value: {expec_reward}")
    except Exception as e:
        print(f"Error processing circuit {j}: {e}")
        rewards[str(ground_truth_index)]["warm_start_qaoa_reward"] = None

with open("circuit_rewards.json", "w") as f:
    json.dump(rewards, f, indent=4)

C:\Users\valte\AppData\Local\Temp\ipykernel_26832\1031935492.py:14: ComplexWarning: Casting complex values to real discards the imaginary part
  coeff = float(coeff)
c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


Expectation Value: -9.492089754261551, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: 3.3861965276897226, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -9.20103178160742, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -12.444773713687201, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -8.516629777751643, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -11.865539894501502, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -10.91280914284775, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -11.329090970824142, Smallest Eigenvalue: -15.999999999999975, Largest Eigenvalue: 55.999999999999915
Expectation Value: -5.0743411997853185

C:\Users\valte\AppData\Local\Temp\ipykernel_26832\1031935492.py:14: ComplexWarning: Casting complex values to real discards the imaginary part
  coeff = float(coeff)
c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


Expectation Value: -8.5, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.499999999999996, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.499999999999993, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.5, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.499999999999998, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.499999999999988, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.500000000000005, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.500000000000016, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.499999999999979, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.500000000000005, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Value: -8.499999999999996, Smallest Eigenvalue: -10.5, Largest Eigenvalue: 10.5
Expectation Valu

C:\Users\valte\AppData\Local\Temp\ipykernel_26832\1031935492.py:14: ComplexWarning: Casting complex values to real discards the imaginary part
  coeff = float(coeff)
c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


Expectation Value: -69.49999999832968, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999681944, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999933932, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999936296, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999491683, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999746515, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.4999999988855, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999813052, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999845885, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999706598, Smallest Eigenvalue: -69.5, Largest Eigenvalue: 135.5
Expectation Value: -69.49999999932828, Smallest Eigenvalue: -69.5, Larg

C:\Users\valte\AppData\Local\Temp\ipykernel_26832\1031935492.py:14: ComplexWarning: Casting complex values to real discards the imaginary part
  coeff = float(coeff)
c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 4
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


Expectation Value: -2.631140815461406, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125Expectation Value: -2.6311408168079664, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125

Expectation Value: -2.6311408225635504, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.631140820292683, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.6311408187539174, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.6311408214215786, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.6311408180685354, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.631140820471902, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.6311408215099963, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.631140816181638, Smallest Eigenvalue: -3.6875, Largest Eigenvalue: 6.3125
Expectation Value: -2.631140809328

KeyError: '3'